In [1]:
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np

import re

# 1.1 Load and Inspect the Data

In [2]:
# display options for better readability
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DB_PATH = Path("data/data.db")
assert DB_PATH.exists(), f"Database not found at {DB_PATH.resolve()}"

# connect to the SQLite database
conn = sqlite3.connect(DB_PATH)

# Retrieve the list of tables in the database
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in Database:")
display(tables)

# Load the target table into a pandas DataFrame, parsing 'ts' as datetime
df = pd.read_sql("SELECT * FROM ohlc_hw", conn, parse_dates=["ts"])
display(df.head())

Tables in Database:


,name
0,ohlc
1,ohlc_hw
2,market_caps
3,ohlc_hw_quarantine
4,ohlc_hw_cleaned


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[us]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  377856 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(1)
memory usage: 27.9 MB


In [4]:
df.describe()

,ts,open,high,low,close,adj_close,volume
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,1998-04-17 21:18:45.292486,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08


### Column Definitions
- **`ts`**: Observation date (timestamp).
- **`ticker`**: Stock ticker symbol. (Note: contains messy HTML tags in the raw data).
- **`open`**: The raw opening price of the stock on that date.
- **`high`**: The raw highest price of the stock on that date.
- **`low`**: The raw lowest price of the stock on that date.
- **`close`**: The raw closing price of the stock on that date.
- **`adj_close`**: The vendor-provided adjusted closing price (accounts for corporate actions like dividends and stock splits).
- **`volume`**: The number of shares traded on that date.

In [5]:
def extract_ticker(raw_ticker):
    '''extract the actual ticker from HTML tags'''
    if pd.isna(raw_ticker):
        return raw_ticker
    match = re.search(r'>([^<]+)<', raw_ticker)
    if match:
        return match.group(1)
    return raw_ticker

print("Date range:", df["ts"].min(), df["ts"].max())
print("Number of tickers:", df["ticker"].nunique())
print("\nTickers (raw):")
print(sorted(df["ticker"].dropna().unique()))
cleaned_preview = df["ticker"].apply(extract_ticker).dropna().unique()
print(sorted(cleaned_preview))

Date range: 1962-01-02 00:00:00 2024-12-06 00:00:00
Number of tickers: 150

Tickers (raw):
['<a target="_blank"><span style="color: blue">AAPL</span></a>', '<a target="_blank"><span style="color: blue">AMGN</span></a>', '<a target="_blank"><span style="color: blue">AMZN</span></a>', '<a target="_blank"><span style="color: blue">AXP</span></a>', '<a target="_blank"><span style="color: blue">BA</span></a>', '<a target="_blank"><span style="color: blue">CAT</span></a>', '<a target="_blank"><span style="color: blue">CRM</span></a>', '<a target="_blank"><span style="color: blue">CSCO</span></a>', '<a target="_blank"><span style="color: blue">CVX</span></a>', '<a target="_blank"><span style="color: blue">DIS</span></a>', '<a target="_blank"><span style="color: blue">GS</span></a>', '<a target="_blank"><span style="color: blue">HD</span></a>', '<a target="_blank"><span style="color: blue">HON</span></a>', '<a target="_blank"><span style="color: blue">IBM</span></a>', '<a target="_blank"><span

# 1.2 Detect Data Quality Issues

In [6]:
def log_issue(issue_list, ticker, issue_type, issue_date, description, severity):
    '''log issues into a standardized list of dictionaries'''
    issue_list.append({
        "ticker": ticker,
        "issue_type": issue_type,
        "issue_date": issue_date,
        "description": description,
        "severity": severity
    })

### Severity Choices

The `severity` column is assigned based on the potential impact on financial analysis and risk modeling:

- **High (Critical)**: These issues indicate mathematically impossible data or extreme statistical anomalies that would immediately break downstream calculations (e.g., calculating returns or volatility).
  - `invalid_price`, `high_lt_close`, `low_gt_close`, `open_out_of_bounds`, `close_out_of_bounds`: Raw OHLC logic is violated. If we use these prices to calculate technical indicators or backtest, the results are completely invalid.
  - `return_outlier`: Returns that exceed 3 standard deviations from the mean suggest either a massive data error (e.g., a missing decimal point) or extreme market movement. If not investigated, this can artificially inflate the portfolio's calculated volatility.

- **Medium (Warning)**: These issues represent missing information but do not inherently introduce false data. 
  - `missing_dates` / `missing_trading_day`: Missing trading days create gaps in the time series. While they don't give you "wrong" prices, they will distort moving averages and compound returns if not handled properly before filling.

- **Low (Informational)**: These are anomalies that might be perfectly legitimate in real-world markets and require contextual awareness rather than automatic action.
  - `zero_volume`: A day with zero trading volume might indicate a halted stock, a market holiday, or a very illiquid micro-cap stock. It's not necessarily a data error, but it's a red flag for liquidity analysis.

In [7]:
# load the reference calendar from the 'ohlc' table
ref_ohlc = pd.read_sql("SELECT ticker, ts FROM ohlc", conn, parse_dates=["ts"])
ref_ohlc["ticker"] = ref_ohlc["ticker"].apply(extract_ticker).str.strip().str.upper()

def detect_missing_dates_with_ref(df, ref_df, ticker_col="ticker", date_col="ts"):
    """
    gaps in the time series by comparing against the reference 'ohlc' table.
    """
    issues = []
    
    # group the reference table by ticker to get the expected dates
    ref_grouped = ref_df.groupby(ticker_col)[date_col].apply(set).to_dict()
    
    # Group the dirty table by ticker
    for ticker, group in df.groupby(ticker_col):
        actual_dates = set(group[date_col].dropna())
        expected_dates = ref_grouped.get(ticker, set())
        
        if not expected_dates:
            continue
            
        #  dates that are in the expected (ref) list but missing in actual list
        # onstrain the check to the min/max range of the actual data
        min_d, max_d = min(actual_dates), max(actual_dates)
        expected_in_range = {d for d in expected_dates if min_d <= d <= max_d}
        
        missing_dates = expected_in_range - actual_dates
        
        if len(missing_dates) > 0:
            sorted_missing = sorted(list(missing_dates))
            start_date = sorted_missing[0].date()
            end_date = sorted_missing[-1].date()
            desc = f"Ticker {ticker} is missing {len(missing_dates)} trading days between {start_date} and {end_date}"
            log_issue(issues, ticker, "missing_dates", None, desc, "medium")
            
            # Log each missing date individually
            for md in sorted_missing:
                log_issue(issues, ticker, "missing_trading_day", md, "Specific trading day missing", "medium")
                
    return issues

print("Demonstrating Missing Dates Detector")
mini_ref = pd.DataFrame({
    "ticker": ["AAPL", "AAPL", "AAPL"],
    "ts": pd.to_datetime(["2020-01-01", "2020-01-02", "2020-01-03"])
})
mini_df = pd.DataFrame({
    "ticker": ["AAPL", "AAPL"],
    "ts": pd.to_datetime(["2020-01-01", "2020-01-03"]) 
})
mini_issues = detect_missing_dates_with_ref(mini_df, mini_ref)
for issue in mini_issues:
    print(issue)

Demonstrating Missing Dates Detector
{'ticker': 'AAPL', 'issue_type': 'missing_dates', 'issue_date': None, 'description': 'Ticker AAPL is missing 1 trading days between 2020-01-02 and 2020-01-02', 'severity': 'medium'}
{'ticker': 'AAPL', 'issue_type': 'missing_trading_day', 'issue_date': Timestamp('2020-01-02 00:00:00'), 'description': 'Specific trading day missing', 'severity': 'medium'}


In [8]:
#detect_other_issues
def detect_other_issues(df, ticker_col="ticker", date_col="ts"):
    issues = []
    
    # We need returns calculated per ticker to find return outliers
    df_sorted = df.sort_values(by=[ticker_col, date_col]).copy()
    df_sorted["prev_close"] = df_sorted.groupby(ticker_col)["close"].shift(1)
    df_sorted["return"] = (df_sorted["close"] - df_sorted["prev_close"]) / df_sorted["prev_close"]
    
    # Calculate mean and std (ddof=1)
    stats = df_sorted.groupby(ticker_col)["return"].agg(["mean", lambda x: x.std(ddof=1)]).reset_index()
    stats.columns = [ticker_col, "mean_ret", "std_ret"]
    
    df_sorted = df_sorted.merge(stats, on=ticker_col, how="left")
    
    for idx, row in df_sorted.iterrows():
        t = row[ticker_col]
        d = row[date_col]
        
        # Return Outliers
        if pd.notna(row["return"]):
            if pd.notna(row["std_ret"]) and row["std_ret"] > 0:
                if abs(row["return"] - row["mean_ret"]) > 3 * row["std_ret"]:
                    log_issue(issues, t, "return_outlier", d, f"Return {row['return']:.4f} is > 3 std from mean {row['mean_ret']:.4f}", "high")
            elif pd.notna(row["std_ret"]) and row["std_ret"] == 0:
                if row["return"] != 0:
                    log_issue(issues, t, "return_outlier", d, "Return differs from 0 but std is 0", "high")
                    
        # `high < close` or `low > close` 
        if pd.notna(row["high"]) and pd.notna(row["close"]):
            if row["high"] < row["close"]:
                log_issue(issues, t, "high_lt_close", d, f"High ({row['high']}) < Close ({row['close']})", "high")
        if pd.notna(row["low"]) and pd.notna(row["close"]):
            if row["low"] > row["close"]:
                log_issue(issues, t, "low_gt_close", d, f"Low ({row['low']}) > Close ({row['close']})", "high")
                
        # nonpositive or non-finite prices
        price_cols = ["open", "high", "low", "close", "adj_close"]
        for col in price_cols:
            if pd.notna(row[col]):
                if row[col] <= 0 or not np.isfinite(row[col]):
                    log_issue(issues, t, "invalid_price", d, f"{col} is nonpositive or non-finite ({row[col]})", "high")
                    
        # Inconsistencies: low <= open <= high and low <= close <= high
        if pd.notna(row["low"]) and pd.notna(row["open"]) and pd.notna(row["high"]):
            if not (row["low"] <= row["open"] <= row["high"]):
                log_issue(issues, t, "open_out_of_bounds", d, f"Open ({row['open']}) outside [{row['low']}, {row['high']}]", "high")
            if not (row["low"] <= row["close"] <= row["high"]):
                log_issue(issues, t, "close_out_of_bounds", d, f"Close ({row['close']}) outside [{row['low']}, {row['high']}]", "high")
                
        # Zero Volume
        if pd.notna(row["volume"]) and row["volume"] == 0:
            log_issue(issues, t, "zero_volume", d, "Volume is exactly 0", "low")
            
    return issues

print("\n Demonstrating Other Issues Detector ")
mini_df_2 = pd.DataFrame({
    "ticker": ["AAPL", "AAPL"],
    "ts": pd.to_datetime(["2020-01-01", "2020-01-02"]),
    "open": [100, 100], "high": [105, 95], "low": [98, 90], "close": [103, 100], "adj_close": [103, 0], "volume": [1000, 0]
})
mini_issues_2 = detect_other_issues(mini_df_2)
for issue in mini_issues_2:
    print(issue)


 Demonstrating Other Issues Detector 
{'ticker': 'AAPL', 'issue_type': 'high_lt_close', 'issue_date': Timestamp('2020-01-02 00:00:00'), 'description': 'High (95) < Close (100)', 'severity': 'high'}
{'ticker': 'AAPL', 'issue_type': 'invalid_price', 'issue_date': Timestamp('2020-01-02 00:00:00'), 'description': 'adj_close is nonpositive or non-finite (0)', 'severity': 'high'}
{'ticker': 'AAPL', 'issue_type': 'open_out_of_bounds', 'issue_date': Timestamp('2020-01-02 00:00:00'), 'description': 'Open (100) outside [90, 95]', 'severity': 'high'}
{'ticker': 'AAPL', 'issue_type': 'close_out_of_bounds', 'issue_date': Timestamp('2020-01-02 00:00:00'), 'description': 'Close (100) outside [90, 95]', 'severity': 'high'}
{'ticker': 'AAPL', 'issue_type': 'zero_volume', 'issue_date': Timestamp('2020-01-02 00:00:00'), 'description': 'Volume is exactly 0', 'severity': 'low'}


In [9]:
df_clean = df.copy()

df_clean["ticker"] = df_clean["ticker"].apply(extract_ticker)
df_clean["ticker"] = df_clean["ticker"].str.strip().str.upper()
# remove exact duplicate rows
exact_dups_before = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
print(f"\nRemoved {exact_dups_before} exact duplicate rows.")

# investigate conflicting (ticker, ts) keys
conflicting_keys = df_clean.duplicated(subset=["ticker", "ts"], keep=False)
num_conflicts = conflicting_keys.sum()

print(f"Found {num_conflicts} rows with conflicting (ticker, ts) keys.")
if num_conflicts > 0:
    print("\n Examples of conflicts:")
    display(df_clean[conflicting_keys].sort_values(by=["ticker", "ts"]).head(6))

print("Running on actual dataset")
all_issues = detect_missing_dates_with_ref(df_clean, ref_ohlc) + detect_other_issues(df_clean)

# Create the final DataFrame
df_quality_report = pd.DataFrame(all_issues)
display(df_quality_report.head(20))

# Summary of issue counts
print("\nIssue Counts")
print(df_quality_report["issue_type"].value_counts())


Removed 29460 exact duplicate rows.
Found 0 rows with conflicting (ticker, ts) keys.
Running on actual dataset


,ticker,issue_type,issue_date,description,severity
0,AAPL,return_outlier,1980-12-26,Return 0.0923 is > 3 std from mean 0.0011,high
1,AAPL,return_outlier,1981-04-02,Return 0.0877 is > 3 std from mean 0.0011,high
2,AAPL,return_outlier,1981-09-25,Return -0.1298 is > 3 std from mean 0.0011,high
3,AAPL,return_outlier,1982-03-19,Return 0.0902 is > 3 std from mean 0.0011,high
4,AAPL,return_outlier,1982-11-22,Return -0.0891 is > 3 std from mean 0.0011,high
5,AAPL,return_outlier,1982-11-30,Return 0.1039 is > 3 std from mean 0.0011,high
6,AAPL,return_outlier,1983-01-20,Return 0.1115 is > 3 std from mean 0.0011,high
7,AAPL,return_outlier,1983-04-20,Return 0.0887 is > 3 std from mean 0.0011,high
8,AAPL,return_outlier,1983-07-26,Return -0.0927 is > 3 std from mean 0.0011,high
9,AAPL,return_outlier,1983-08-31,Return 0.1331 is > 3 std from mean 0.0011,high



Issue Counts
issue_type
return_outlier        5056
zero_volume             75
invalid_price            5
open_out_of_bounds       5
Name: count, dtype: int64


# 1.3 Clean the Data

***Outlier Repair Strategy:**
For impossible price relationships (e.g., `high < close`), I chose **Option 1: Replace with NaN then forward-fill**. 

**Justification:**
If the fundamental OHLC logic is violated for a specific day, it implies the entire record for that day is corrupted. Setting the raw prices to `NaN` and forward-filling with the previous valid day's prices ensures we eliminate mathematically impossible bounds.
*Trade-off Acknowledgement:* While forward-filling prevents logical errors within the corrupted day, it creates a "flat" price, which may cause an artificially large return spike on the subsequent valid day.

In [10]:
def handle_missing_values(df, ref_df):
    """
    Handle missing values
      - Drop rows where ALL raw price columns are missing (before filling).
      - Forward-fill missing raw prices within ticker groups.
      - Fill missing volume with 0 and flag it (unknown != no trading).
      - Reconstruct missing/invalid adj_close from validated raw close
        times a date-specific factor matched on (ticker, ts) from the
        supplied reference table.
      - Quarantine rows whose adj_close still cannot be resolved.
    """
    df_out = df.copy()
    df_out["volume_repaired"] = False
    df_out["price_repaired"] = False
    df_out["adj_close_repaired"] = False
    raw_prices = ["open", "high", "low", "close"]

    # Drop rows where ALL raw prices are missing (before filling)
    all_missing_mask = df_out[raw_prices].isna().all(axis=1)
    print(f"Dropping {all_missing_mask.sum()} rows where all raw prices are missing.")
    df_out = df_out[~all_missing_mask].copy()

    # Missing volume: 0, keep repair flag
    vol_missing = df_out["volume"].isna()
    df_out.loc[vol_missing, "volume"] = 0
    df_out.loc[vol_missing, "volume_repaired"] = True

    # Forward-fill raw prices within ticker
    # Sort first so ffill carries the correct previous observation.
    df_out = df_out.sort_values(["ticker", "ts"]).reset_index(drop=True)
    for col in raw_prices:
        missing_before = df_out[col].isna()
        df_out[col] = df_out.groupby("ticker")[col].ffill()
        df_out.loc[missing_before & df_out[col].notna(), "price_repaired"] = True

    #Build trusted date-specific factor from VALID reference rows
    # Only rows with positive finite close and adj_close contribute; a dirty
    # reference close would otherwise make the factor explode.
    ref = ref_df.copy()
    ref_ok = (
        np.isfinite(ref["close"]) & (ref["close"] > 0)
        & np.isfinite(ref["adj_close"]) & (ref["adj_close"] > 0)
    )
    ref = ref[ref_ok].copy()
    ref["trusted_factor"] = ref["adj_close"] / ref["close"]
    # Plausible-range sanity check: exclude pathological adjustment ratios.
    ref = ref[ref["trusted_factor"].between(0.01, 100)]

    df_out = df_out.merge(
        ref[["ticker", "ts", "trusted_factor"]],
        on=["ticker", "ts"], how="left"
    )
    hit_rate = df_out["trusted_factor"].notna().mean()
    print(f"factor merge hit rate: {hit_rate:.1%}")

    #Rebuild adj_close where it is missing / nonpositive / non-finite
    adj_bad = (
        df_out["adj_close"].isna()
        | (df_out["adj_close"] <= 0)
        | ~np.isfinite(df_out["adj_close"])
    )
    can_rebuild = adj_bad & df_out["trusted_factor"].notna() & df_out["close"].notna()
    df_out.loc[can_rebuild, "adj_close"] = (
        df_out.loc[can_rebuild, "close"] * df_out.loc[can_rebuild, "trusted_factor"]
    )
    df_out.loc[can_rebuild, "adj_close_repaired"] = True

    # Quarantine rows whose adj_close is still unresolvable
    adj_still_bad = (
        df_out["adj_close"].isna()
        | (df_out["adj_close"] <= 0)
        | ~np.isfinite(df_out["adj_close"])
    )
    # Preserve any pre-existing quarantine flags if present.
    if "quarantine" not in df_out.columns:
        df_out["quarantine"] = False
    df_out["quarantine"] = df_out["quarantine"].fillna(False) | adj_still_bad
    print(f"adj_close still bad & quarantined: {int(adj_still_bad.sum())}")

    # trusted_factor is intentionally kept for handle_outliers().
    return df_out

def repair_adj_close_spikes(
    df, min_abs_return=0.30, symmetry_tol=0.25, raw_disagreement_frac=0.50
):
    """
    Repair isolated one-day adjusted-close spikes.

    A spike is a day t where the adjusted-close log return from t-1 to t
    and from t to t+1 are both large, opposite in sign, and nearly cancel.
    The raw close must not show the same pattern, which separates a bad
    vendor adjusted-close value from a real two-day market reversal.

    Confirmed spikes are rebuilt as close * trusted_factor on the matching
    (ticker, ts) date. If no trusted factor exists, the row is quarantined
    rather than extrapolated.
    """
    df_out = df.copy()
    if "adj_close" not in df_out.columns:
        return df_out

    if "adj_close_spike_flag" not in df_out.columns:
        df_out["adj_close_spike_flag"] = False
    if "adj_close_spike_action" not in df_out.columns:
        df_out["adj_close_spike_action"] = "none"
    if "adj_close_repaired" not in df_out.columns:
        df_out["adj_close_repaired"] = False
    if "quarantine" not in df_out.columns:
        df_out["quarantine"] = False

    has_factor = "trusted_factor" in df_out.columns
    finite_pos = lambda s: np.isfinite(s) & s.gt(0)

    adj = df_out["adj_close"].where(finite_pos(df_out["adj_close"]))
    raw = df_out["close"].where(finite_pos(df_out["close"]))

    with np.errstate(divide="ignore", invalid="ignore"):
        adj_ret = np.log(adj / adj.groupby(df_out["ticker"]).shift(1))
        raw_ret = np.log(raw / raw.groupby(df_out["ticker"]).shift(1))
    adj_ret = adj_ret.replace([np.inf, -np.inf], np.nan)
    raw_ret = raw_ret.replace([np.inf, -np.inf], np.nan)

    df_out["adj_ret"] = adj_ret
    df_out["adj_ret_next"] = df_out.groupby("ticker")["adj_ret"].shift(-1)
    df_out["raw_ret"] = raw_ret
    df_out["raw_ret_next"] = df_out.groupby("ticker")["raw_ret"].shift(-1)

    return_spike = (
        df_out["adj_ret"].notna()
        & df_out["adj_ret_next"].notna()
        & df_out["adj_ret"].abs().gt(min_abs_return)
        & df_out["adj_ret_next"].abs().gt(min_abs_return)
        & (df_out["adj_ret"] * df_out["adj_ret_next"]).lt(0)
        & (df_out["adj_ret"] + df_out["adj_ret_next"]).abs().le(
            symmetry_tol * df_out[["adj_ret", "adj_ret_next"]].abs().max(axis=1)
        )
        & df_out["raw_ret"].abs().lt(raw_disagreement_frac * df_out["adj_ret"].abs())
        & df_out["raw_ret_next"].abs().lt(
            raw_disagreement_frac * df_out["adj_ret_next"].abs()
        )
    )
    if has_factor:
        factor_bad = (
            finite_pos(df_out["close"])
            & finite_pos(df_out["trusted_factor"])
            & finite_pos(df_out["adj_close"])
            & (df_out["adj_close"]
               / (df_out["close"] * df_out["trusted_factor"])
               - 1).abs().gt(1.0)
        )
    else:
        factor_bad = pd.Series(False, index=df_out.index)
    valid = return_spike | factor_bad
    df_out["adj_close_spike_flag"] = valid

    if has_factor:
        can_rebuild = (
            valid & df_out["close"].notna() & df_out["trusted_factor"].notna()
        )
    else:
        can_rebuild = pd.Series(False, index=df_out.index)

    df_out.loc[can_rebuild, "adj_close"] = (
        df_out.loc[can_rebuild, "close"] * df_out.loc[can_rebuild, "trusted_factor"]
    )
    df_out.loc[can_rebuild, "adj_close_spike_action"] = (
        "repaired_from_trusted_factor"
    )
    df_out.loc[can_rebuild, "adj_close_repaired"] = True

    unresolved = valid & ~can_rebuild
    df_out.loc[unresolved, "adj_close_spike_action"] = "quarantine_unresolved"
    df_out["quarantine"] = (
        df_out["quarantine"].fillna(False).astype(bool) | unresolved
    )

    df_out = df_out.drop(
        columns=["adj_ret", "adj_ret_next", "raw_ret", "raw_ret_next"],
        errors="ignore",
    )
    print(
        f"Adjusted-close spikes detected: {int(valid.sum())}; "
        f"rebuilt {int(can_rebuild.sum())}; quarantined {int(unresolved.sum())}."
    )
    return df_out

def handle_outliers(df):
    """
    Repair impossible OHLC relationships and investigate extreme returns.

      - Impossible price relationships: clear only the offending field(s) and
        forward-fill within ticker (Option 1 from the assignment). The chosen
        option is documented in this docstring.
      - Extreme returns (> 3 sigma, both tails): classify each flagged row
        into an explicit action. Only confirmed data errors are interpolated
        between valid neighbors; extreme returns without supporting evidence
        are KEPT and FLAGGED, never auto-deleted.
      - Interpolation is restricted to internal gaps (no leading/trailing
        extrapolation)
      - All raw bounds and adjusted-price consistency are rechecked after
        repair.
    """
    df_out = df.copy()
    raw_prices = ["open", "high", "low", "close"]
    finite_pos = lambda s: np.isfinite(s) & s.gt(0)

    # Preserve quarantine flags from handle_missing_values.
    if "quarantine" not in df_out.columns:
        df_out["quarantine"] = False
    else:
        df_out["quarantine"] = df_out["quarantine"].fillna(False).astype(bool)

    # Ensure helper flag columns exist and are boolean.
    if "price_repaired" not in df_out.columns:
        df_out["price_repaired"] = False
    if "adj_close_repaired" not in df_out.columns:
        df_out["adj_close_repaired"] = False

    # impossible OHLC bounds
    # Option 1 (documented): clear the offending field(s) to NaN, then ffill.
    close_oob = df_out["close"].notna() & (
        df_out["close"].lt(df_out["low"]) | df_out["close"].gt(df_out["high"])
    )
    open_oob = df_out["open"].notna() & (
        df_out["open"].lt(df_out["low"]) | df_out["open"].gt(df_out["high"])
    )
    range_bad = df_out["high"].lt(df_out["low"])

    print(
        f"Found close_oob={close_oob.sum()}, open_oob={open_oob.sum()}, "
        f"range_bad={range_bad.sum()}. Clearing offending fields then ffilling."
    )

    df_out.loc[close_oob, "close"] = np.nan
    df_out.loc[open_oob, "open"] = np.nan
    df_out.loc[range_bad, ["high", "low"]] = np.nan
    df_out["price_repaired"] |= close_oob | open_oob | range_bad

    for col in raw_prices:
        df_out[col] = df_out.groupby("ticker")[col].ffill()

    # extreme return detection
    df_out["extreme_return_flag"] = False
    df_out["extreme_return_action"] = "none"
    df_out["consecutive_extreme"] = False

    valid_close = df_out["close"].where(finite_pos(df_out["close"]))
    df_out["return"] = valid_close.groupby(df_out["ticker"]).pct_change(
        fill_method=None
    )

    grp = df_out.groupby("ticker")["return"]
    mean_r = grp.transform("mean")
    std_r = grp.transform("std")
    # Sample std (ddof=1) is pandas default; explicit std_r.gt(0) handles
    # the zero/undefined-std case.
    df_out["extreme_return_flag"] = (
        ((df_out["return"] - mean_r).abs() > 3 * std_r) & std_r.gt(0)
    )

    # Consecutive extremes: two adjacent flagged returns with the same sign
    # usually indicate one corrupted close distorting two returns.
    def _consecutive(group):
        flag = group["extreme_return_flag"].fillna(False)
        sgn = np.sign(group["return"].fillna(0))
        prev_flag = flag.shift(1, fill_value=False)
        prev_sgn = sgn.shift(1, fill_value=0)
        both = flag & prev_flag & (sgn == prev_sgn) & (sgn != 0)
        return both | both.shift(-1, fill_value=False).astype(bool)

    df_out["consecutive_extreme"] = (
        df_out.groupby("ticker", group_keys=False)
        .apply(_consecutive, include_groups=False)
        .fillna(False)
        .astype(bool)
    )

    repair_series = df_out["price_repaired"].fillna(False)
    prev_repair = repair_series.groupby(df_out["ticker"]).shift(1).fillna(False)
    df_out["return_affected_by_repair"] = (
        df_out["extreme_return_flag"] & (repair_series | prev_repair)
    )

    # Confirmed close error: outside today's [low, high] or non-finite/non-positive.
    df_out["confirmed_close_error"] = (
        ~finite_pos(df_out["close"])
        | df_out["close"].lt(df_out["low"])
        | df_out["close"].gt(df_out["high"])
        | df_out["high"].lt(df_out["low"])
        | df_out["high"].lt(df_out[["open", "close"]].max(axis=1))
        | df_out["low"].gt(df_out[["open", "close"]].min(axis=1))
    )

    # Corporate-action candidate: raw extreme but adjusted NOT extreme.
    # We record this as a candidate only; no split/merger is asserted.
    if "adj_close" in df_out.columns:
        adj_valid = df_out["adj_close"].where(finite_pos(df_out["adj_close"]))
        df_out["adj_return"] = adj_valid.groupby(df_out["ticker"]).pct_change(
            fill_method=None
        )
        ag = df_out.groupby("ticker")["adj_return"]
        adj_extreme = (
            ((df_out["adj_return"] - ag.transform("mean")).abs()
             > 3 * ag.transform("std"))
            & ag.transform("std").gt(0)
        )
        df_out["corporate_action_candidate"] = (
            df_out["extreme_return_flag"]
            & ~df_out["return_affected_by_repair"]
            & ~adj_extreme.fillna(False)
        )
    else:
        df_out["corporate_action_candidate"] = False

    #  classify extreme returns 
    is_ext = df_out["extreme_return_flag"]
    repair_aff = df_out["return_affected_by_repair"]
    corp_cand = df_out["corporate_action_candidate"]
    confirmed_err = df_out["confirmed_close_error"]
    consec = df_out["consecutive_extreme"]

    df_out.loc[is_ext & repair_aff, "extreme_return_action"] = (
        "keep_flag_return_affected_by_repair"
    )
    df_out.loc[is_ext & ~repair_aff & corp_cand, "extreme_return_action"] = (
        "keep_flag_corporate_action_candidate"
    )
    df_out.loc[
        is_ext & ~repair_aff & ~corp_cand & confirmed_err,
        "extreme_return_action",
    ] = "interpolate_if_internal_else_quarantine"
    df_out.loc[
        is_ext & ~repair_aff & ~corp_cand & ~confirmed_err & consec,
        "extreme_return_action",
    ] = "quarantine_consecutive_extreme"
    df_out.loc[
        is_ext & ~repair_aff & ~corp_cand & ~confirmed_err & ~consec,
        "extreme_return_action",
    ] = "keep_flag_unexplained"

    # interpolate confirmed internal errors 
    to_fill = (
        df_out["extreme_return_action"].eq(
            "interpolate_if_internal_else_quarantine"
        )
        & ~repair_series
    )
    df_out["close_for_fill"] = df_out["close"]
    df_out.loc[to_fill, "close_for_fill"] = np.nan

    prev_valid = df_out.groupby("ticker")["close_for_fill"].ffill()
    next_valid = df_out.groupby("ticker")["close_for_fill"].bfill()
    interp = df_out.groupby("ticker")["close_for_fill"].transform(
        lambda s: s.interpolate(method="linear", limit_area="inside")
    )

    can_interp = (
        to_fill & interp.notna() & prev_valid.notna() & next_valid.notna()
    )
    df_out.loc[can_interp, "close"] = interp[can_interp]
    df_out.loc[can_interp, "price_repaired"] = True
    df_out.loc[to_fill & ~can_interp, "quarantine"] = True
    df_out.loc[
        can_interp & ~df_out["close"].between(df_out["low"], df_out["high"]),
        "quarantine",
    ] = True
    df_out.loc[
        df_out["extreme_return_action"].eq("quarantine_consecutive_extreme"),
        "quarantine",
    ] = True

    #keep adj_close consistent after close repair 
    # Rebuild adj_close ONLY for rows where close was changed by a repair and
    # a trusted date-specific factor exists. Original vendor adj_close on
    # unrepaired rows is preserved as-is (preserve the vendor convention).
    if {"adj_close", "trusted_factor"} <= set(df_out.columns):
        rebuilt_now = (
            (can_interp | df_out["price_repaired"].fillna(False))
            & df_out["trusted_factor"].notna()
            & df_out["close"].notna()
        )
        df_out.loc[rebuilt_now, "adj_close"] = (
            df_out.loc[rebuilt_now, "close"] * df_out.loc[rebuilt_now, "trusted_factor"]
        )
        df_out.loc[rebuilt_now, "adj_close_repaired"] = True

    # Repair isolated adjusted-close spikes after raw-close classification,
    # so this step does not change raw-return/quarantine decisions.
    df_out = repair_adj_close_spikes(df_out)

    # Residual NaN in raw prices after ffill -> quarantine.
    residual_nan = df_out[raw_prices].isna().any(axis=1) & ~df_out["quarantine"]
    df_out.loc[residual_nan, "quarantine"] = True

    # post-repair verification
    # These are the checks the assignment explicitly requires.
    ok = ~df_out["quarantine"].fillna(False)
    #finite and positive raw prices on retained rows
    assert np.isfinite(df_out.loc[ok, raw_prices]).all().all()
    assert df_out.loc[ok, raw_prices].gt(0).all().all()
    #valid OHLC bounds on retained rows
    assert df_out.loc[ok, "high"].ge(df_out.loc[ok, "low"]).all()
    assert df_out.loc[ok, "close"].between(
        df_out.loc[ok, "low"], df_out.loc[ok, "high"]
    ).all()
    assert df_out.loc[ok, "open"].between(
        df_out.loc[ok, "low"], df_out.loc[ok, "high"]
    ).all()
    # adjusted-price consistency: finite & positive everywhere
    assert np.isfinite(df_out.loc[ok, "adj_close"]).all()
    assert (df_out.loc[ok, "adj_close"] > 0).all()

    if {"trusted_factor", "adj_close_repaired"} <= set(df_out.columns):
        chk = (
            ok
            & finite_pos(df_out["close"])
            & finite_pos(df_out["adj_close"])
            & df_out["trusted_factor"].notna()
            & df_out["adj_close_repaired"].fillna(False)
        )
        n_chk = int(chk.sum())
        if n_chk:
            lhs = df_out.loc[chk, "adj_close"]
            rhs = df_out.loc[chk, "close"] * df_out.loc[chk, "trusted_factor"]
            if not np.allclose(lhs, rhs, rtol=1e-6, atol=1e-8):
                rel = (lhs - rhs).abs() / rhs.abs().clip(lower=1e-12)
                worst = rel.sort_values(ascending=False).head(5)
                print("adj_close rebuild mismatch on repaired rows (top 5):")
                print(worst.to_string())
                raise AssertionError(
                    f"{int((rel > 1e-6).sum())} repaired rows violate "
                    "adj_close = close * trusted_factor"
                )
            print(f"adj_close rebuild check passed on {n_chk} repaired rows.")

    #  Audit log
    print(f"Flagged {int(df_out['extreme_return_flag'].sum())} extreme returns.")
    if df_out["extreme_return_flag"].any():
        print(
            df_out.loc[df_out["extreme_return_flag"], "extreme_return_action"]
            .value_counts()
            .to_string()
        )
    if df_out["quarantine"].any():
        print(
            f"Quarantined {int(df_out['quarantine'].sum())} rows; "
            "exclude them from downstream analysis."
        )

    return df_out

In [11]:
print("Starting data cleaning process")

df_clean = df_clean.sort_values(by=["ticker", "ts"]).reset_index(drop=True)
n_before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["ticker", "ts"], keep="first")
print(f"Dropped {n_before_dedup - len(df_clean)} exact duplicate (ticker, ts) rows.")

# Load reference table
ref_ohlc_full = pd.read_sql(
    "SELECT ticker, ts, close, adj_close FROM ohlc",
    conn, parse_dates=["ts"]
)
ref_ohlc_full["ticker"] = (
    ref_ohlc_full["ticker"].apply(extract_ticker).str.strip().str.upper()
)

#ts dtypes must match or factor merge silently fails
print("df_clean.ts dtype:", df_clean["ts"].dtype)
print("ref.ts dtype:", ref_ohlc_full["ts"].dtype)
assert pd.api.types.is_datetime64_any_dtype(df_clean["ts"]), \
    "df_clean['ts'] must be datetime64 for the (ticker, ts) factor merge"
assert pd.api.types.is_datetime64_any_dtype(ref_ohlc_full["ts"]), \
    "ref_ohlc_full['ts'] must be datetime64"

# Run cleaning pipeline
df_clean = handle_missing_values(df_clean, ref_ohlc_full)
df_clean = handle_outliers(df_clean)

print("\nData cleaning and repair completed.")


# Account for retained / removed / quarantined rows
n_raw_loaded      = len(df)                 # original dirty table (captured at load)
n_after_dedup     = len(df_clean)           # still includes quarantined rows
n_quarantined     = int(df_clean["quarantine"].fillna(False).sum())
n_retained_pre    = n_after_dedup - n_quarantined

# Keep the raw table unchanged, read its row count directly for audit
n_raw_in_db = pd.read_sql("SELECT COUNT(*) AS n FROM ohlc_hw", conn)["n"].iloc[0]

print("\n row accounting")
print(f"Raw table 'ohlc_hw' rows in DB            : {int(n_raw_in_db)}")
print(f"Rows loaded for cleaning                  : {n_raw_loaded}")
print(f"Rows after exact (ticker, ts) dedup       : {n_after_dedup}")
print(f"Removed by dedup                          : {n_raw_loaded - n_after_dedup}")
print(f"Quarantined (unrepairable)                : {n_quarantined}")
print(f"Retained (to be written to cleaned table) : {n_retained_pre}")


# Exclude quarantined rows; archive them separately for audit
if "quarantine" in df_clean.columns and n_quarantined:
    quar = df_clean.loc[df_clean["quarantine"].fillna(False)].copy()
    # Keep quarantine rows with a reason column for auditability
    quar["quarantine_reason"] = np.where(
        quar["extreme_return_action"].eq("quarantine_consecutive_extreme"),
        "consecutive_extreme",
        np.where(
            quar["adj_close"].isna() | (quar["adj_close"] <= 0) | ~np.isfinite(quar["adj_close"]),
            "adj_close_unresolved",
            "other_unrepairable",
        ),
    )
    quar["ts"] = pd.to_datetime(quar["ts"]).dt.strftime("%Y-%m-%d")
    # Do not persist the internal factor into the audit table
    quar_save = quar.drop(columns=["trusted_factor", "close_for_fill",
                                    "return", "adj_return"], errors="ignore")
    quar_save.to_sql("ohlc_hw_quarantine", conn, if_exists="replace", index=False)
    print(f"Saved {len(quar_save)} quarantined rows to 'ohlc_hw_quarantine'.")

df_final = df_clean.loc[~df_clean["quarantine"].fillna(False)].copy()

# Store ts consistently as ISO dates (YYYY-MM-DD)
df_final["ts"] = pd.to_datetime(df_final["ts"]).dt.strftime("%Y-%m-%d")

# Save repair flags with the cleaned data
# Keep: volume_repaired, price_repaired, adj_close_repaired,
#       extreme_return_flag, extreme_return_action, quarantine (all False here)
# Drop: intermediate helpers that are not part of the deliverable
helper_cols_to_drop = [
    "trusted_factor", "close_for_fill", "return", "adj_return",
    "return_affected_by_repair", "confirmed_close_error",
    "corporate_action_candidate", "consecutive_extreme",
]
df_final = df_final.drop(columns=helper_cols_to_drop, errors="ignore")

# Retained rows may still contain leading-gap NaNs that ffill could not fill.
# Those are NOT quarantined by handle_outliers; drop them here and account.
n_before_nan_drop = len(df_final)
df_final = df_final.dropna(subset=["open", "high", "low", "close", "adj_close"])
n_leading_dropped = n_before_nan_drop - len(df_final)
print(f"Dropped {n_leading_dropped} retained rows with unresolved leading gaps.")
print(f"Final retained rows                      : {len(df_final)}")


# Verify: no missing / non-finite / nonpositive values in retained
#       raw and adjusted price columns; finite nonnegative volume
print("\n Final validation")
raw_prices = ["open", "high", "low", "close"]
price_cols = raw_prices + ["adj_close"]
check_cols = price_cols + ["volume"]

# No missing values
missing = df_final[check_cols].isna().sum()
print("Missing (NaN) counts:")
print(missing.to_string())
print(f"   total NaN: {int(missing.sum())}")
assert missing.sum() == 0, "Retained rows contain NaN in required columns"

# No non-finite values
nonfinite = (~np.isfinite(df_final[check_cols].astype(float))).sum()
print("Non-finite (inf/-inf) counts:")
print(nonfinite.to_string())
print(f"   total non-finite: {int(nonfinite.sum())}")
assert nonfinite.sum() == 0, "Retained rows contain non-finite values"

# No nonpositive prices
nonpositive = (df_final[price_cols] <= 0).sum()
print(" Nonpositive price counts (<= 0):")
print(nonpositive.to_string())
print(f"   total nonpositive: {int(nonpositive.sum())}")
assert nonpositive.sum() == 0, "Retained rows contain nonpositive prices"

# volume finite and nonnegative
vol = df_final["volume"].astype(float)
print(" Volume checks:")
print(f"   negative volume rows : {int((vol < 0).sum())}")
print(f"   non-finite volume rows: {int((~np.isfinite(vol)).sum())}")
assert (vol >= 0).all(), "Negative volume in retained rows"
assert np.isfinite(vol).all(), "Non-finite volume in retained rows"


# [nique (ticker, ts) keys
print("\n Unique (ticker, ts) keys")
dup_mask = df_final.duplicated(subset=["ticker", "ts"], keep=False)
n_dup_rows = int(dup_mask.sum())
n_dup_groups = int(df_final.loc[dup_mask, ["ticker", "ts"]]
                   .drop_duplicates().shape[0])
print(f"   duplicated rows          : {n_dup_rows}")
print(f"   duplicated (ticker, ts) groups: {n_dup_groups}")
assert n_dup_rows == 0, "Duplicate (ticker, ts) keys in retained rows"


# Verify: all price relationships are valid
print("\n OHLC relationship validation")
tol = 1e-8
low_  = df_final["low"].astype(float)
high_ = df_final["high"].astype(float)

invalid_bounds = (
    (df_final["open"]  < low_  - tol) |
    (df_final["open"]  > high_ + tol) |
    (df_final["close"] < low_  - tol) |
    (df_final["close"] > high_ + tol) |
    (high_ < low_ - tol)
)
n_invalid = int(invalid_bounds.sum())
print(f"   rows with invalid OHLC relationships: {n_invalid}")
assert n_invalid == 0, "Invalid OHLC relationships in retained rows"

#  Adjusted-price consistency after any raw-price repair
# The exact per-row adj_close = close * trusted_factor check on repaired
# rows is already performed inside handle_outliers(). 
repaired_mask = df_final["price_repaired"] | df_final["adj_close_repaired"]
print("\n Adjusted-price consistency after repair:")
print(f"   repaired rows retained: {int(repaired_mask.sum())}")
assert np.isfinite(df_final.loc[repaired_mask, "adj_close"].astype(float)).all()
assert (df_final.loc[repaired_mask, "adj_close"].astype(float) > 0).all()
print("   retained repaired adj_close values are finite and positive.")


# Show before/after statistics
print("\n Before / after statistics")
before_stats = df[raw_prices + ["adj_close", "volume"]].describe().T
after_stats  = df_final[raw_prices + ["adj_close", "volume"]].astype(float).describe().T

print("\n BEFORE (raw dirty input)")
display(before_stats[["count", "mean", "min", "max"]].round(4))
print("\n AFTER (retained cleaned rows) ")
display(after_stats[["count", "mean", "min", "max"]].round(4))

print("\n Row accounting summary")
summary = pd.DataFrame([
    ("Raw rows in 'ohlc_hw'",            int(n_raw_in_db)),
    ("Rows loaded",                      int(n_raw_loaded)),
    ("Removed by exact dedup",           int(n_raw_loaded - n_after_dedup)),
    ("Quarantined (unrepairable)",       int(n_quarantined)),
    ("Removed as leading-gap NaN",       int(n_leading_dropped)),
    ("Final retained rows",              int(len(df_final))),
    ("Sum check (retained+removed+quar)",
     int(len(df_final) + n_quarantined + n_leading_dropped
         + (n_raw_loaded - n_after_dedup))),
], columns=["Stage", "Rows"])
display(summary)


# [Create new table 'ohlc_hw_cleaned' in the database
#       (raw 'ohlc_hw' left untouched)
table_name = "ohlc_hw_cleaned"
df_final.to_sql(table_name, conn, if_exists="replace", index=False)
n_in_db = pd.read_sql(
    f"SELECT COUNT(*) AS n FROM {table_name}", conn
)["n"].iloc[0]
print(f"\n Successfully saved to table '{table_name}'.")
print(f"      Rows written          : {int(n_in_db)}")
print(f"      Raw table 'ohlc_hw'   : unchanged ({int(n_raw_in_db)} rows)")

# Verify ts stored as ISO YYYY-MM-DD in DB
ts_sample = pd.read_sql(
    f"SELECT ts FROM {table_name} ORDER BY ts LIMIT 3", conn
)
print(f"      ts sample (ISO dates) : {ts_sample['ts'].tolist()}")
assert ts_sample["ts"].str.match(r"^\d{4}-\d{2}-\d{2}$").all(), \
    "ts column is not stored as ISO YYYY-MM-DD"

print("\nDone.")

Starting data cleaning process
Dropped 0 exact duplicate (ticker, ts) rows.
df_clean.ts dtype: datetime64[us]
ref.ts dtype: datetime64[us]
Dropping 0 rows where all raw prices are missing.
factor merge hit rate: 100.0%
adj_close still bad & quarantined: 0
Found close_oob=0, open_oob=5, range_bad=0. Clearing offending fields then ffilling.
Adjusted-close spikes detected: 10622; rebuilt 10622; quarantined 0.
adj_close rebuild check passed on 28975 repaired rows.
Flagged 5056 extreme returns.
extreme_return_action
keep_flag_corporate_action_candidate    4960
keep_flag_unexplained                     85
quarantine_consecutive_extreme            11
Quarantined 16 rows; exclude them from downstream analysis.

Data cleaning and repair completed.

 row accounting
Raw table 'ohlc_hw' rows in DB            : 397711
Rows loaded for cleaning                  : 397711
Rows after exact (ticker, ts) dedup       : 368251
Removed by dedup                          : 29460
Quarantined (unrepairable)     

,count,mean,min,max
open,397711.0,4.628550e+01,0.0000,6.042600e+02
high,397711.0,4.674770e+01,0.0052,6.086300e+02
low,397711.0,4.581630e+01,0.0048,5.978800e+02
close,397711.0,4.629290e+01,0.0052,6.054000e+02
adj_close,377856.0,5.165230e+01,0.0029,1.044251e+04
volume,397711.0,3.097773e+07,0.0000,9.276606e+09



 AFTER (retained cleaned rows) 


,count,mean,min,max
open,368235.0,4.631010e+01,0.0052,6.042600e+02
high,368235.0,4.677250e+01,0.0052,6.086300e+02
low,368235.0,4.584110e+01,0.0048,5.978800e+02
close,368235.0,4.631800e+01,0.0052,6.054000e+02
adj_close,368235.0,3.727510e+01,0.0029,6.054000e+02
volume,368235.0,3.094967e+07,0.0000,9.276606e+09



 Row accounting summary


,Stage,Rows
0,Raw rows in 'ohlc_hw',397711
1,Rows loaded,397711
2,Removed by exact dedup,29460
3,Quarantined (unrepairable),16
4,Removed as leading-gap NaN,0
5,Final retained rows,368235
6,Sum check (retained+removed+quar),397711



 Successfully saved to table 'ohlc_hw_cleaned'.
      Rows written          : 368235
      Raw table 'ohlc_hw'   : unchanged (397711 rows)
      ts sample (ISO dates) : ['1962-01-02', '1962-01-02', '1962-01-02']

Done.


In [12]:
df_final

,ts,open,high,low,close,adj_close,volume,ticker,volume_repaired,price_repaired,adj_close_repaired,quarantine,extreme_return_flag,extreme_return_action,adj_close_spike_flag,adj_close_spike_action
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,False,False,False,False,False,none,False,none
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,False,False,False,False,False,none,False,none
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,False,False,False,False,False,none,False,none
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,False,False,False,False,False,none,False,none
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,False,False,False,False,False,none,False,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
368246,2024-10-14,80.05000,80.51000,79.76000,80.29000,80.290000,9914500,WMT,False,False,False,False,False,none,False,none
368247,2024-10-15,80.67000,81.82000,80.51000,81.65000,81.650000,13276221,WMT,False,False,False,False,False,none,False,none
368248,2024-10-16,81.53000,81.66000,80.62000,81.22000,81.220000,11123300,WMT,False,False,False,False,False,none,False,none
368249,2024-10-17,81.27000,81.50000,80.65000,80.89000,80.890000,9424705,WMT,False,False,False,False,False,none,False,none
